# Interactive Code Notebook 

...

---



This following code implements a **Visual Saliency Detection Engine** using the **Spectral Residual (SR) approach**.

In computer vision, **visual saliency** predicts where a human eye would naturally focus when looking at an image (e.g., bright spots, anomalous textures, distinct objects against a background). This specific algorithm is based on the landmark 2007 paper by Xiaodi Hou and Liqing Zhang: *["Saliency Detection: A Spectral Residual Approach"](https://cvpr.org)*.

---

## The Theoretical Core: Spectral Residual Model

The central premise of the Spectral Residual model stems from **Information Theory**:

$$\text{Image Information} = \text{Prior Knowledge (Background)} + \text{Innovation (Saliency)}$$

Most images share statistical similarities in the frequency domain (the "background" or redundant noise). If you subtract this average, predictable background profile from the image's actual spectrum, what remains is the **spectral residual**—the unpredictable, high-information regions that attract human attention.

```
       [ Input Image ]
              │
              ▼
   [ 2D Fourier Transform ] ───► Extract Amplitude & Phase
              │
              ├───► Log Amplitude ──► Smooth (Local Average)
              │           │                    │
              │           └─────────┬──────────┘
              │                     ▼
              │            Subtract Average
              │                     │
              │                     ▼
              │            [ Spectral Residual ]
              │                     │
              └───► Combine with original Phase
                            │
                            ▼
               [ Inverse 2D Fourier Transform ]
                            │
                            ▼
               [ Gaussian Blur & Scale [0, 1] ]
                            │
                            ▼
                     [ Saliency Heatmap ]

```

---

## Line-by-Line Code Breakdown

### 1. Initialization and Input Preprocessing

```python
def __init__(self, processing_dim: tuple = (128, 128)):
    self.processing_dim = processing_dim

```

* **Purpose:** Sets a fixed downsampled resolution. Frequency domain analysis is scale-invariant and computationally heavy; reducing the image to $128 \times 128$ speeds up the Fast Fourier Transform (FFT) dramatically while preserving broad spatial structures.

```python
# Note: There is a minor bug in the provided code on this line:
original_h, original_w = img_bgr[:2] 

```

* **Conversion:** Converts the standard OpenCV BGR image into a 1-channel grayscale image ($I(x,y)$), because luminance carries most visual saliency cues.

---

### 2. Fast Fourier Transform & Frequency Extraction

```python
small = cv2.resize(gray, self.processing_dim, interpolation = cv2.INTER_AREA).astype(np.float32)
fft = np.fft.fft2(small)
amplitude = np.abs(fft)
phase = np.angle(fft)

```

The 2D Discrete Fourier Transform converts the image from the spatial domain $(x, y)$ to the frequency domain $(u, v)$:

$$\mathcal{F}(u, v) = R(u, v) + i \cdot I(u, v)$$

From this complex output, two key components are calculated:

1. **Magnitude / Amplitude Spectrum $\vert{}A(f)\vert{}$:** Measures how much of each spatial frequency is present.
2. **Phase Spectrum $P(f)$:** Contains the positional and structural information about where those frequencies exist in space.

> **Crucial Insight:** The phase spectrum $P(f)$ holds the location of visual features. Reconstructing an image using its phase with a constant or flattened amplitude still yields identifiable object boundaries.

---

### 3. Calculating the Spectral Residual

```python
log_amp = np.log(amplitude + 1e-8)
avg_log_amp = cv2.blur(log_amp, (3, 3))
spectral_residual = log_amp - avg_log_amp

```

* **`np.log(amplitude + 1e-8)`:** Natural logarithm $A(f) = \ln(\vert{}FFT\vert{})$ of the amplitude spectrum. The scale of spatial frequencies follows a steep power-law curve ($1/f$). Taking the logarithm linearizes this curve, making it easier to analyze. `1e-8` is added to prevent $\ln(0)$ errors.
* **`cv2.blur(log_amp, (3, 3))`:** Applies a $3 \times 3$ box filter to compute the local average logarithmic amplitude, denoted as $h_n(f) * A(f)$. This represents the "redundant background spectrum."
* **`spectral_residual = log_amp - avg_log_amp`:** Subtracts the smoothed spectrum from the log amplitude:

$$R(f) = A(f) - h_n(f) * A(f)$$

$R(f)$ is the **Spectral Residual**—the non-redundant signal that represents unexpected visual features.

---

### 4. Reconstructing the Spatial Saliency Energy

```python
combined = np.exp(spectral_residual + 1j * phase)
saliency = np.abs(np.fft.ifft2(combined)) ** 2

```

1. **Re-combining Magnitude & Phase:** Euler's formula states $e^{a + ib} = e^a e^{ib} = e^a (\cos b + i \sin b)$.
* `np.exp(spectral_residual)` restores the residual magnitude back to linear scale.
* `1j * phase` injects the original structural phase back into the signal.


2. **Inverse FFT (`ifft2`):** Transforms the residual frequency signal back into spatial coordinates $(x, y)$.
3. **`np.abs(...) ** 2`:** Taking the squared magnitude gives the **Saliency Energy**:

$$S(x,y) = \left\vert{} \mathcal{F}^{-1} \left( \exp(R(f) + i \cdot P(f)) \right) \right\vert{}^2$$

---

### 5. Post-Processing & Normalization

```python
saliency = cv2.GaussianBlur(saliency.astype(np.float32), (9, 9), sigmaX = 2.5)
saliency = cv2.resize(saliency, (original_w, original_h), interpolation = cv2.INTER_CUBIC)

```

* **`cv2.GaussianBlur`:** Human visual acuity drops off gradually from the center of focus. Smoothing the energy map with a Gaussian filter ($\sigma = 2.5$, kernel = $9 \times 9$) aggregates scattered high-frequency noise into smooth, coherent regions of interest.
* **`cv2.resize`:** Upscales the small $128 \times 128$ saliency map back to match the original image size using bicubic interpolation.

```python
min_v, max_v = np.min(saliency), np.max(saliency)

if max_v > min_v:
    heatmap = (saliency - min_v) / (max_v - min_v)
else:
    heatmap = np.zeros_like(saliency)

```

* **Min-Max Scaling:** Scales all pixel values into the normalized range $[0.0, 1.0]$. The most salient point becomes `1.0` (white) and the least salient becomes `0.0` (black).

---

### 6. Extracting the Point of Highest Interest

```python
def get_peak_salient_point(self, heatmap: np.ndarray) -> tuple[int, int]:
    _, _, _, max_loc = cv2.minMaxLoc(heatmap)
    return max_loc # returns (x, y)

```

* **`cv2.minMaxLoc`:** Finds the coordinate with the maximum value in the normalized matrix. This $(x, y)$ coordinate represents the primary focal point of the image (e.g., where a camera system should center its auto-focus or bounding box).

---

## Practical Applications

* **Object Detection / Tracking:** Pre-filtering image frames to find regions of interest before running heavy neural networks (YOLO, Faster-RCNN).
* **Autofocus Systems:** Automatically selecting the primary focus area in digital cameras.
* **Image Compression:** Allocating higher bitrates/quality to salient regions while aggressively compressing predictable background areas.
* **UI/UX Heatmaps:** Predicting where users will look first on a webpage or graphic design layout.

In [ ]:
import cv2
import numpy as np 

class SaliencyEngine:

    """
    This class will compute the visual saliency (heatmap) of a given image
    """

    def __init__(self, processing_dim: tuple = (128, 128)):
        self.processing_dim = processing_dim


    def compute_heatmap(self, img_bgr: np.ndarray) -> np.ndarray:

        """
        Calculates an unconstrained spectral residual saliency heatmap; i.e. in the range [0.0, 1.0]
        """

        original_h, original_w = img_bgr.shape[:2]
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

        #Downsample the image for global frequency analysis (ENCODING)
        small = cv2.resize(gray, self.processing_dim, interpolation = cv2.INTER_AREA).astype(np.float32)

        #Calculate the (Fast) Fourier Transform to obtain the Log Amplitude and Phase
        fft = np.fft.fft2(small)
        amplitude = np.abs(fft)
        phase = np.angle(fft)

        log_amp = np.log(amplitude + 1e-8)
        avg_log_amp = cv2.blur(log_amp, (3, 3))
        spectral_residual = log_amp - avg_log_amp

        #Inverse FFT to get the Saliency Energy
        combined = np.exp(spectral_residual + 1j * phase)
        saliency = np.abs(np.fft.ifft2(combined)) ** 2

        #Spatial Gaussian Smoothing (to simulate visual acuity spread)
        saliency = cv2.GaussianBlur(saliency.astype(np.float32), (9, 9), sigmaX = 2.5)

        #Upsample (DECODE) back to native frame dimensions and normalise to [0, 1]
        saliency = cv2.resize(saliency, (original_w, original_h), interpolation = cv2.INTER_CUBIC)

        min_v, max_v = np.min(saliency), np.max(saliency)

        if max_v > min_v:
            heatmap = (saliency - min_v) / (max_v - min_v)

        else:
            heatmap = np.zeros_like(saliency)
    

        return heatmap


    def get_peak_salient_point(self, heatmap: np.ndarray) -> tuple[int, int]:

        """
        Finds the most salient point, represented in (x, y); where (x, y) = argmax S(x, y)
        """

        _, _, _, max_loc = cv2.minMaxLoc(heatmap)
        return max_loc # i.e. (x,y)

    

This Python file implements a **Foveal Extraction and Gaze Normalization Engine**.

In biology and computer vision, **foveation** mimics the human visual system: while the peripheral vision senses the broader environment (often processed via low-resolution saliency maps), the **fovea**—the central region of the retina—extracts a high-resolution sub-region around the target for fine-grained feature analysis.

This class takes the target focal point $(cx, cy)$—such as the peak salient coordinate produced by the previous `SaliencyEngine`—crops a fixed-size square patch around it while preventing out-of-bounds errors, and converts the pixel coordinates into a normalized **gaze vector**.

---

## The Theoretical Core: Biological Foveation & Gaze Normalization

This module bridges global low-resolution visual processing with localized high-resolution perception through two key concepts:

```
    [ Full Image (Width x Height) ]
   ┌──────────────────────────────┐
   │                              │
   │        (cx, cy) Target       │
   │           ┌───────┐          │
   │           │ Crop  │          │  ───►  [ High-Res Foveal Crop ] (96 x 96)
   │           └───────┘          │
   │                              │
   └──────────────────────────────┘
                  │
                  ▼
   [ Coordinate Normalization ]   ───►  Gaze Vector [norm_x, norm_y] in [-1.0, 1.0]

```

### 1. Foveal Cropping & Boundary Clamping

To extract a square patch of size $S \times S$ centered at an arbitrary coordinate $(cx, cy)$, the window spans from $cx - \frac{S}{2}$ to $cx + \frac{S}{2}$ horizontally and $cy - \frac{S}{2}$ to $cy + \frac{S}{2}$ vertically.

If the focus point lies near the image edge, a naive slice will extend past the array boundaries, causing array truncation or out-of-bounds crashes. To prevent this, the center point is constrained (**clamped**) into a safe operational bounding region:

$$x_{\text{clamped}} = \min(\max(cx, \text{half\_size}), \text{width} - \text{half\_size})$$

$$y_{\text{clamped}} = \min(\max(cy, \text{half\_size}), \text{height} - \text{half\_size})$$

### 2. Zero-Centered Gaze Normalization

Pixel coordinates are dependent on image resolution and origin offset (top-left vs. center). For downstream machine learning models (e.g., reinforcement learning agents, gaze trackers, neural networks), spatial coordinates are normalized into a scale-invariant interval $[-1.0, 1.0]$:

* $(-1.0, -1.0)$ represents the top-left corner.
* $(0.0, 0.0)$ represents the exact geometric center of the image frame.
* $(1.0, 1.0)$ represents the bottom-right corner.

---

## Line-by-Line Code Breakdown

### 1. Initialization

```python
def __init__(self, crop_size: int = 96):
    self.crop_size = crop_size
    self.half_size = crop_size // 2

```

* **`crop_size = 96`:** Defines the width and height of the foveal square patch in pixels. $96 \times 96$ is a standard input shape for lightweight convolutional networks (such as MobileNet or custom CNN backbones).
* **`half_size = crop_size // 2`:** Computes the radial offset (48 pixels) from the center to any edge of the crop box using integer division.

---

### 2. Boundary Clamping & Cropping

```python
height, width = img_rgb.shape[:2]

# Clamp the centre coordinate so that the crop remains within image bounds
clamped_x = int(np.clip(cx, self.half_size, width - self.half_size))
clamped_y = int(np.clip(cy, self.half_size, height - self.half_size))

```

* **`np.clip(a, a_min, a_max)`:** Restricts $cx$ to lie strictly within $[\text{half\_size}, \text{width} - \text{half\_size}]$ and $cy$ within $[\text{half\_size}, \text{height} - \text{half\_size}]$.
* **Effect:** If a salient point is detected at $(x=5, y=10)$ in a $640 \times 480$ image, `clamped_x` shifts to `48` and `clamped_y` shifts to `48`. This ensures that slicing `[48 - 48 : 48 + 48]` produces a full $96 \times 96$ crop without spilling past pixel index `0`.

```python
crop = img_rgb[
    clamped_y - self.half_size : clamped_y + self.half_size,
    clamped_x - self.half_size : clamped_x + self.half_size,
]

```

* **NumPy Slicing:** Extracts the spatial sub-region from the input image using standard matrix indexing `[row_start : row_end, col_start : col_end]`. The output dimensions are guaranteed to be exact: $(\text{crop\_size}, \text{crop\_size}, 3)$.

---

### 3. Coordinate Normalization (Gaze Projection)

```python
# Normalise the gaze point to [-1.0, 1.0] such that point 0.0 is the visual centre
norm_x = (cx / float(width)) * 2.0 - 1.0
norm_y = (cy / float(height)) * 2.0 - 1.0

gaze_coords = np.array([norm_x, norm_y], dtype = np.float32)

```

The linear transformation converts standard image pixel space $[0, W] \times [0, H]$ to normalized coordinate space $[-1, 1] \times [-1, 1]$:

$$\text{norm\_x} = 2 \cdot \left(\frac{cx}{W}\right) - 1$$

$$\text{norm\_y} = 2 \cdot \left(\frac{cy}{H}\right) - 1$$

#### Mathematical Mapping Examples:

| Input Position $(cx, cy)$ | Normalized Output $(\text{norm\_x}, \text{norm\_y})$ | Visual Meaning |
| --- | --- | --- |
| $(0, 0)$ | $(-1.0, -1.0)$ | Top-Left Corner |
| $(W/2, H/2)$ | $(0.0, 0.0)$ | Image Center |
| $(W, H)$ | $(1.0, 1.0)$ | Bottom-Right Corner |

Notice that the raw target coordinates $(cx, cy)$ are used for `gaze_coords` (not the clamped ones). This retains the true, unconstrained direction where the visual engine wants to look, even if the crop box itself had to stop at the image edge.

---

## Practical Applications

* **Robotic Vision & Active Vision Systems:** Mimics eye saccades. A camera captures a wide-field image, identifies an object of interest, crops the high-resolution target for classification, and tracks its normalized motor steering angles $(x, y)$.
* **Foveated Rendering:** Used in Virtual Reality (VR/AR) headsets with eye tracking to render full resolution only where the user is looking, drastically lowering GPU load.
* **Deep Reinforcement Learning Agents:** Reduces neural network computational complexity by feeding a small, constant tensor ($96 \times 96 \times 3$) and two positional scalars $[x, y]$ instead of processing high-resolution video streams in full.

In [ ]:
import cv2 
import numpy as np

class FovealExtractor:

    """
    Extracts a bounded foveal crop around the most salient point of 
    an image and normalises the gaze
    """

    def __init__(self, crop_size: int = 96):
        self.crop_size = crop_size
        self.half_size = crop_size // 2


    def extract(self, img_rgb: np.ndarray, cx: int, cy: int) -> tuple[np.ndarray, np.ndarray]:

        height, width = img_rgb.shape[:2]

        #Clamp the centre coordinate so that the crop remains within image bounds
        clamped_x = int(np.clip(cx, self.half_size, width - self.half_size))
        clamped_y = int(np.clip(cy, self.half_size, height - self.half_size))

        crop = img_rgb[
            clamped_y - self.half_size : clamped_y + self.half_size,
            clamped_x - self.half_size : clamped_x + self.half_size,
        ]

        #Normalise the gaze point to [-1.0, 1.0] such that point 0.0 is the visual centre
        norm_x = (cx / float(width)) * 2.0 - 1.0
        norm_y = (cy / float(height)) * 2.0 - 1.0

        gaze_coords = np.array([norm_x, norm_y], dtype = np.float32)

        return crop, gaze_coords 

This following code implements a **Foveal Vision Encoder using Deep Feature Extraction**.

In the overall visual architecture, this module acts as the **feature representation layer**: it takes the high-resolution cropped foveal image and the 2D gaze coordinates produced by the previous stages (`FovealExtractor`), extracts deep semantic visual features using a pre-trained **ResNet-18** deep learning backbone, and fuses them into a single **514-dimensional latent state vector**.

---

## The Theoretical Core: Multimodal Feature Fusion & Transfer Learning

```
  [ Foveal Crop (96x96x3) ]           [ Gaze Coordinates [norm_x, norm_y] ]
             │                                          │
             ▼                                          │
  [ ImageNet Normalization ]                            │
             │                                          │
             ▼                                          │
  [ Frozen ResNet-18 Backbone ]                         │
  (Conv Layers ──► Layer 4 ──► Adaptive AvgPool)        │
             │                                          │
             ▼                                          │
   [ 512-d Visual Features ]                            │
             │                                          │
             └───────────────────┬──────────────────────┘
                                 ▼
                     [ Feature Concatenation ]
                                 │
                                 ▼
                 [ Output State Vector (514-d) ]
                 [ 512 Visual + 2 Positional ]

```

### 1. Transfer Learning with a Pre-trained Backbone

Instead of training a neural network from scratch, the encoder leverages a **ResNet-18** architecture pre-trained on ImageNet.

* **Feature Extraction:** Pre-trained Convolutional Neural Networks (CNNs) act as powerful universal feature extractors. Early convolutional layers detect low-level primitives (edges, textures), while deeper layers (`layer3`, `layer4`) compress high-level semantic concepts (shapes, object parts, distinct patterns).
* **Dimensionality Reduction:** The network compresses a $96 \times 96 \times 3$ pixel patch ($27,648$ raw values) into an abstract, semantically dense $512$-dimensional vector.

### 2. Feature Fusion (Visual + Spatial Context)

Visual information alone ("what" is seen) lacks spatial context ("where" it is located). By concatenating the **512 visual features** with the **2 normalized gaze coordinates** $(x, y)$, the output vector creates a unified **multimodal state space** containing both semantic and spatial representation:

$$\mathbf{z} = \begin{bmatrix} \mathbf{v}_{\text{resnet}} \\ x_{\text{gaze}} \\ y_{\text{gaze}} \end{bmatrix} \in \mathbb{R}^{514}$$

---

## Line-by-Line Code Breakdown

### 1. Initialization and Model Truncation

```python
resnet = models.resnet18(weights = models.ResNet18_Weights.DEFAULT)

```

* **Pre-trained Weights:** Loads standard ResNet-18 weights trained on ImageNet ($1.2$ million images across $1000$ classes).

```python
# Truncate at the avgpool to remove the 1000-class classification head 
self.backbone = nn.Sequential(
    resnet.conv1,
    resnet.bn1,
    resnet.relu,
    resnet.maxpool, 
    resnet.layer1,
    resnet.layer2,
    resnet.layer3,
    resnet.layer4,
    resnet.avgpool,
    nn.Flatten(),
).to(self.device)

```

* **Truncation:** Standard ResNet-18 ends with a fully connected layer (`resnet.fc`) mapping features to $1000$ class probabilities. By dropping `resnet.fc` and retaining everything up to `resnet.avgpool` followed by `nn.Flatten()`, the feature extractor produces a flat feature vector of shape `(batch_size, 512)`.

---

### 2. Weight Freezing & Evaluation Mode

```python
# Freeze the encoder weights
self.backbone.eval()

for param in self.backbone.parameters():
    param.requires_grad = False

```

* **`requires_grad = False`:** Disables gradient computation for all layers in the backbone. This saves GPU memory and speeds up forward passes since backpropagation graph tracking is turned off.
* **`self.backbone.eval()`:** Sets layers like `BatchNorm` to evaluation mode so they use pre-calculated running statistics instead of batch statistics.

---

### 3. Preprocessing Transformation Pipeline

```python
# Note: There is a typo in the original code ('T.compose' instead of 'T.Compose')
self.transform = T.Compose([
    T.ToTensor(), 
    T.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225]),
])

```

* **`T.ToTensor()`:** Converts a NumPy uint8 RGB image array of shape `(H, W, C)` in range $[0, 255]$ into a PyTorch float tensor of shape `(C, H, W)` in range $[0.0, 1.0]$.
* **`T.Normalize()`:** Standardizes color channels using standard ImageNet mean and standard deviation:

$$I_{\text{normalized}} = \frac{I - \text{mean}}{\text{std}}$$

---

### 4. Encoding Step (Forward Pass & Fusion)

```python
tensor = self.transform(crop_rgb).unsqueeze(0).to(self.device)

```

* **`unsqueeze(0)`:** Transforms a single image tensor of shape `(3, H, W)` into a 4D batch tensor of shape `(1, 3, H, W)` required by PyTorch CNN layers.

```python
with torch.no_grad():
    features = self.backbone(tensor).squeeze(0).cpu().numpy()

```

* **`torch.no_grad()`:** Context manager ensuring no execution graph is built.
* **`squeeze(0).cpu().numpy()`:** Strips the batch dimension, returning a 1D NumPy array of length `512`.

```python
# Concatenate: [512 visual latent features, norm_x, and norm_y]
return np.concatenate([features, gaze_coords], axis = 0).astype(np.float32)

```

* **Vector Fusion:** Joins the 512-dimensional visual vector with the 2-element gaze array, returning a final 1D array of length **514**.

---

## Technical Notes & Fixes

1. **Typo in `T.compose`:** PyTorch's composition function is capitalized as `T.Compose`.
2. **Device Handling during Eval Mode:** Placing `self.backbone.eval()` inside `__init__` sets evaluation mode on creation. However, if a user later calls `model.train()` on an outer wrapper class, `eval()` should be reinforced inside the `encode()` function to prevent `BatchNorm` layers from updating running statistics during execution.

---

## Complete Pipeline Integration

Together, the three modules form an end-to-end active vision system:

1. **`SaliencyEngine` (Global Perception):** Uses Fourier Spectral Residual analysis to find *where* the most interesting visual stimulus is located in the image.
2. **`FovealExtractor` (Local Attention):** Crops a high-resolution sub-region around that coordinate and converts the pixel location into normalized gaze coordinates.
3. **`FovealVisionEncoder` (Semantic Feature Extraction):** Translates the raw crop and gaze coordinates into a 514-dimensional representation suitable for downstream machine learning tasks (e.g., Reinforcement Learning policies, visual navigation, or active target tracking).

In [ ]:
import torch 
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T 
import numpy as np


class FovealVisionEncoder(nn.Module):

    """
    Processes foveal crops through a frozen ResNet-18 backbone into a 514-d vector
    """

    def __init__(self, device: torch.device):

        super().__init__()
        self.device = device

        resnet = models.resnet18(weights = models.ResNet18_Weights.DEFAULT)

        #Truncate at the avgpool to remove the 1000-class classification head 
        self.backbone = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool, 
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
            resnet.avgpool,
            nn.Flatten(),
        ).to(self.device)

        #Freeze the encoder weights
        self.backbone.eval()

        for param in self.backbone.parameters():
            param.requires_grad = False


        self.transform = T.compose([T.ToTensor(), T.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225]),])

        #This is just to highlight that we'll have:
            # output state dimension = 512 visual features
            # + 2 gaze coordinates 
            # Therefore, is 514 in total

        self.output_dim = 512 + 2


    def encode(self, crop_rgb: np.ndarray, gaze_coords: np.ndarray) -> np.ndarray:

        tensor = self.transform(crop_rgb).unsqueeze(0).to(self.device)

        with torch.no_grad():
            features = self.backbone(tensor).squeeze(0).cpu().numpy()

        #Concatenate: [512 visual latent features, norm_x, and norm_y]
        return np.concatenate([features, gaze_coords], axis = 0).astype(np.float32)


    

The following code ties all three previous components into a unified **Reinforcement Learning Environment Wrapper** using Farama Gymnasium (formerly OpenAI Gym).

In reinforcement learning (RL) algorithms like **SAC (Soft Actor-Critic)**, processing high-resolution raw pixel matrices directly as environment states requires massive neural networks and consumes huge amounts of memory and compute. This wrapper acts as a **foveated perception filter**: it intercepts raw visual observations from an environment, runs them through the active vision pipeline, and converts high-dimensional video streams into a concise **514-dimensional continuous vector** for the RL agent.

---

## The Theoretical Core: Active Vision in Reinforcement Learning

```
           [ Gymnasium Environment / Screen Grabber ]
                               │
                               ▼
                        [ Raw RGB Frame ]
                               │
            ┌──────────────────┴──────────────────┐
            ▼                                     ▼
 [ Saliency Engine (Frequency) ]        [ Foveal Extractor ]
            │                                     │
            ▼                                     ▼
 [ Peak Salient Point (x, y) ]  ───►  [ High-Res Patch Crop ]
                                                  │
                                                  ▼
                                    [ ResNet-18 Vision Encoder ]
                                                  │
                                                  ▼
                                      [ 514-d State Vector ]
                                                  │
                                                  ▼
                                       [ RL Agent Policy (SAC) ]

```

### 1. The ObservationWrapper Pattern

In Gymnasium, an `ObservationWrapper` modifies the observation space before it reaches the RL agent policy without altering the environment's underlying physics or reward logic.

Every time the environment runs a step (`env.step()`), this wrapper intercepts the output frame, runs it through the image-to-state transformation pipeline, and yields a standardized 1D vector.

### 2. Information Compression & Dimensionality Reduction

* **Raw Image Input:** A standard $1080\text{p}$ RGB frame consists of $1920 \times 1080 \times 3 \approx 6.22 \times 10^6$ parameters.
* **Foveated Representation:** The active vision wrapper strips redundant background information and compresses the input down to **514 continuous values**—a **$12,000\times$ compression factor**—while preserving both semantic visual content and spatial focal context.

---

## Line-by-Line Code Breakdown

### 1. Initialization and Space Definition

```python
class SaliencyFoveatedWrapper(gym.ObservationWrapper):
    def __init__(self, env: gym.Env, window_grabber, crop_size: int = 96, device: str = None):
        super().__init__(env)
        self.grabber = window_grabber
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))

```

* **`super().__init__(env)`:** Registers the base Gymnasium environment.
* **`window_grabber`:** An external frame acquisition module (e.g., a screen-capture instance or camera interface) used to pull raw image frames.
* **`self.device`:** Automatically routes neural network evaluation to an Nvidia GPU (`cuda`) if available, otherwise defaulting to the CPU.

```python
self.saliency_engine = UnbiasedSaliencyEngine()
self.cropper = FovealExtractor(crop_size = crop_size)
self.encoder = FovealVisionEncoder(device = self.device)

```

* **Sub-module Instantiation:** Initializes all three pipeline components studied in the previous steps.

```python
self.observation_space = Box(
    low = -np.inf,
    high = np.inf,
    shape = (self.encoder.output_dim,), # Note: Shape expects a tuple
    dtype = np.float32
)

```

* **`gym.spaces.Box`:** Defines the mathematical bounds and shape of the observation space for the RL policy network.
* **`shape`:** Set to `(514,)` (retrieved from `self.encoder.output_dim`).
* **`low / high`:** Set to $[-\infty, \infty]$ because ResNet feature vectors and normalized gaze coordinates contain unbounded or negative float values.



```python
self.latest_frame_rgb = None
self.latest_heatmap = None
self.latest_gaze = (0, 0)
self.latest_crop = None

```

* **Telemetry & Debugging Cache:** Stores intermediate pipeline arrays in memory. This allows real-time visualization of the agent's "visual attention" (plotting heatmaps, gaze points, and crops on a UI) during training without re-running computations.

---

### 2. The Observation Transformation Pipeline

```python
def observation(self, obs) -> np.ndarray:
    # Get the image
    frame_bgr = self.grabber.grab_bgr()
    frame_bgr = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

```

* **Frame Grab:** Ignores the raw `obs` parameter from the base environment and instead grabs the latest raw BGR frame from `window_grabber`.
* **Color Conversion:** Converts BGR to standard RGB for consistency with PyTorch's ImageNet normalization requirements.

```python
    # Get Saliency Map & Peak Fixation Point
    heatmap = self.saliency_engine.compute_heatmap(frame_bgr)
    peak_x, peak_y = self.saliency_engine.get_peak_salient_point(heatmap)

    # Apply Foveation Crop to Most Salient Part
    crop_rgb, gaze_coords = self.cropper.extract(frame_bgr, peak_x, peak_y)

```

* **Active Vision Loop:**
1. Computes the Fourier-based visual saliency map (`compute_heatmap`).
2. Identifies the peak focus location `(peak_x, peak_y)`.
3. Crops a high-resolution $96 \times 96$ patch (`crop_rgb`) and extracts the zero-centered gaze coordinates (`gaze_coords`).



```python
    # Extract info for inspection
    self.latest_frame_rgb = frame_bgr
    self.latest_heatmap = heatmap
    self.latest_gaze = (peak_x, peak_y)
    self.latest_crop = crop_rgb

    rl_state = self.encoder.encode(crop_rgb, gaze_coords)
    return rl_state

```

* **State Generation:** Caches the intermediate visual assets for debugging, feeds the foveal crop and gaze coordinates into the ResNet-18 encoder, and returns the final 1D `rl_state` tensor of shape `(514,)` to the RL agent.

---

## Technical Notes & Bounding Fixes

1. **Shape Tuple Bug in `Box` Space:**
```python
# Original:
shape = (self.encoder.output_dim) # Evaluates to integer 514

# Fix:
shape = (self.encoder.output_dim,) # Evaluates to tuple (514,)

```


2. **Color Channel Consistency Note:**
In `observation()`, `frame_bgr` is converted to RGB and passed directly into `compute_heatmap()`. However, standard OpenCV functions expect BGR. Ensure that `UnbiasedSaliencyEngine` expects RGB input or perform conversion after computing saliency.

---

## Summary of the Full System Architecture

This wrapper completes an **end-to-end biologically inspired visual cognitive architecture**:

| Component | Biological Equivalent | Mathematical / CS Role | Output |
| --- | --- | --- | --- |
| **`UnbiasedSaliencyEngine`** | Superior Colliculus / Peripheral Vision | 2D Fourier Spectral Residual Analysis | Saliency Heatmap & Focus Point $(x,y)$ |
| **`FovealExtractor`** | Fovea & Retinal Mapping | Spatial Cropping & Coordinate Mapping | $96 \times 96$ Crop & Gaze Vector $[-1, 1]^2$ |
| **`FovealVisionEncoder`** | Visual Cortex (V1–V4 / IT) | Deep Transfer Learning (ResNet-18) | 512-d Visual + 2-d Gaze Vector |
| **`SaliencyFoveatedWrapper`** | Thalamocortical Loop | Gymnasium Environment State Filter | 514-d Vector to RL Agent (SAC) |

In [ ]:
import gymnasium as gym
from gymnasium.spaces import Box
import torch
from visual_processing.fixation_locator import UnbiasedSaliencyEngine
from visual_processing.foveal_extractor import FovealExtractor
from visual_processing.vision_feature_encoder import FovealVisionEncoder
import numpy as np
import cv2



class SaliencyFoveatedWrapper(gym.ObservationWrapper):

    """
    Gym wrapper to encapsulate the active vision. The pipeline is as follows:

    Image -> Unbiased Saliency Map -> Peak Argmax
    -> Foveal Crop -> RestNet-18 -> SAC State
    """

    def __init__(self, env: gym.Env, window_grabber, crop_size: int = 96, device: str = None):

        super().__init__(env)
        self.grabber = window_grabber
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))

        self.saliency_engine = UnbiasedSaliencyEngine()
        self.cropper = FovealExtractor(crop_size = crop_size)
        self.encoder = FovealVisionEncoder(device = self.device)

        #Gym Observation state should be 514-dimensional (see notebook)
        self.observation_space = Box(
            low = -np.inf,
            high = np.inf,
            shape = (self.encoder.output_dim),
            dtype = np.float32
        )

        self.latest_frame_rgb = None
        self.latest_heatmap = None
        self.latest_gaze = (0, 0)
        self.latest_crop = None


    def observation(self, obs) -> np.ndarray:

        #Get the image
        frame_bgr = self.grabber.grab_bgr()
        frame_bgr = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

        #Get Saliency Map & Peak Fixation Point
        heatmap = self.saliency_engine.compute_heatmap(frame_bgr)
        peak_x, peak_y = self.saliency_engine.get_peak_salient_point(heatmap)

        #Apply Foveation Crop to Most Salient Part
        crop_rgb, gaze_coords = self.cropper.extract(frame_bgr, peak_x, peak_y)

        #Extract info for inspection
        self.latest_frame_rgb = frame_bgr
        self.latest_heatmap = heatmap
        self.latest_gaze = (peak_x, peak_y)
        self.latest_crop = crop_rgb

        rl_state = self.encoder.encode(crop_rgb, gaze_coords)
        return rl_state


This Python file implements a **Verification, Diagnostics, and Visualization Harness** for the entire active vision architecture.

While the previous three files built the core components (`SaliencyEngine`, `FovealExtractor`, `FovealVisionEncoder`, and `SaliencyFoveatedWrapper`), this script acts as an end-to-end unit test and visual debugger. It validates that the wrapped environment produces the expected 514-dimensional state vectors and renders a side-by-side diagnostic dashboard using `matplotlib` to inspect what the agent sees.

---

## The Theoretical Core: Diagnostic Inspection in Active Vision RL

When training Reinforcement Learning agents on complex vision tasks, debugging high-dimensional vectors (like a 514-element state array) directly is nearly impossible. This test harness serves two main purposes:

1. **Dimensionality & Pipeline Validation:** Ensures that the environment correctly wraps the underlying Gymnasium `env`, initializes without shape mismatches, and outputs a flat state vector of length $514$.
2. **Visual Inspection of Internal Telemetry:** Exposes the internal state of the visual processing pipeline to confirm that:
* Saliency highlights meaningful, high-information regions.
* The focal point $(gx, gy)$ aligns with the peak intensity of the heatmap.
* The $96 \times 96$ foveal crop accurately centers on the target without distortion or boundary clipping.



```
       [ Environment Reset & Step Loop ]
                      │
                      ▼
            [ Intercept Telemetry ]
   ┌──────────────────┼──────────────────┐
   ▼                  ▼                  ▼
[ Full Frame ]  [ Heatmap ]     [ Foveal Crop ]
   │                  │                  │
   └──────────────────┼──────────────────┘
                      ▼
         [ Matplotlib 1x3 Dashboard ]

```

---

## Line-by-Line Code Breakdown

### 1. Harness Setup and Initialization

```python
def test_unbiased_active_vision(env, grabber):
    wrapped_env = SaliencyFoveatedWrapper(env, grabber, crop_size=96)

    # 1. Reset
    state = wrapped_env.reset()
    print(f"Observation space: {wrapped_env.observation_space.shape}")
    print(f"Sampled state shape: {state.shape} (Expected: 514)")

```

* **Environment Wrapping:** Instantiates the `SaliencyFoveatedWrapper` with a $96 \times 96$ crop window using the base Gymnasium environment and frame grabber.
* **`wrapped_env.reset()`:** Resets the environment state.
* **Validation Output:** Prints the environment's observation space and the actual output shape. This verifies that the 512 ResNet features and 2 gaze coordinates are correctly fused into a 1D vector of shape `(514,)`.

---

### 2. Stepping through the Environment

```python
# 2. Step forward 5 steps
for step in range(1, 6):
    action = np.array([0.0, 0.5, -1.0])
    state, reward, done, info = wrapped_env.step(action)

    print(f"Step {step} | Peak Fixation: {wrapped_env.latest_gaze}")

```

* **Control Loop:** Simulates $5$ consecutive environment interaction steps using a static dummy continuous action vector `[0.0, 0.5, -1.0]` (typical for continuous control tasks like car driving or robotic manipulation).
* **Fixation Telemetry:** Reads `wrapped_env.latest_gaze` at each step to log the pixel coordinates $(gx, gy)$ where the active vision system decided to focus.

---

### 3. Matplotlib Visualization Dashboard

```python
if step == 5:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

```

* On the 5th step, the script creates a side-by-side $1 \times 3$ grid of plots to visualize the complete visual processing pipeline:

#### Panel 1: Original RGB Image + Fixation Marker

```python
gx, gy = wrapped_env.latest_gaze
axes[0].imshow(wrapped_env.latest_frame_rgb)
axes[0].scatter(
    [gx],
    [gy],
    c="red",
    s=120,
    marker="x",
    linewidths=2.5,
    label=f"Peak Saliency ({gx}, {gy})",
)
axes[0].set_title("Input Frame + Saliency Argmax")
axes[0].legend(loc="lower right")
axes[0].axis("off")

```

* **Role:** Renders the uncompressed, full-resolution RGB frame (`latest_frame_rgb`).
* **Overlay:** Places a red "X" marker directly on top of the image at coordinate $(gx, gy)$, visually proving where the saliency algorithm placed its attention.

#### Panel 2: Spectral Residual Saliency Heatmap

```python
axes[1].imshow(wrapped_env.latest_heatmap, cmap="inferno")
axes[1].set_title("Unbiased Saliency Heatmap")
axes[1].axis("off")

```

* **Role:** Displays the 2D floating-point energy map generated by Fourier Spectral Residual Analysis.
* **`cmap="inferno"`:** Uses the `inferno` colormap (black/purple for low-saliency background, bright yellow/white for peak attention regions) to make intensity gradients easily readable.

#### Panel 3: Foveal Crop

```python
axes[2].imshow(wrapped_env.latest_crop)
axes[2].set_title("Foveated Crop (Sent to ResNet-18)")
axes[2].axis("off")

```

* **Role:** Shows the exact $96 \times 96 \times 3$ RGB tensor that gets passed into the ResNet-18 vision backbone. This confirms that clamping logic worked and the patch contains the target feature centered within the crop window.

---

### 4. Step Termination Handling

```python
if done:
    break

```

* Standard safety check: exits the loop early if the environment reaches a terminal state before 5 steps complete.

---

## Technical Notes & Gymnasium API Adjustments

Depending on the version of Gymnasium used in your environment setup, keep the following API adjustments in mind:

1. **`reset()` Tuple Unpacking:**
In modern Gymnasium (v0.26+), `env.reset()` returns a tuple `(observation, info)` instead of just `observation`:
```python
# Gymnasium standard:
state, info = wrapped_env.reset()

```


2. **`step()` Tuple Unpacking:**
Gymnasium uses a 5-element return tuple `(observation, reward, terminated, truncated, info)` rather than the older 4-element Gym return `(state, reward, done, info)`:
```python
# Gymnasium standard:
state, reward, terminated, truncated, info = wrapped_env.step(action)
done = terminated or truncated

```



---

## Complete Pipeline Summary Across All 4 Files

With this final file, the entire system architecture is mapped out from raw pixels to model execution:

```
┌─────────────────────────────────────────────────────────────────────────────────┐
│ 1. SaliencyEngine                                                              │
│    Frequency Domain Analysis ──► Log-Amplitude Residual ──► Peak (x, y)          │
└────────────────────────────────────────┬────────────────────────────────────────┘
                                         │
                                         ▼
┌─────────────────────────────────────────────────────────────────────────────────┐
│ 2. FovealExtractor                                                              │
│    Boundary Clamping ──► $96\times96$ Crop ──► Normalized Gaze Vector $[-1, 1]^2$  │
└────────────────────────────────────────┬────────────────────────────────────────┘
                                         │
                                         ▼
┌─────────────────────────────────────────────────────────────────────────────────┐
│ 3. FovealVisionEncoder                                                          │
│    Frozen ResNet-18 Backbone ──► 512-d Visual Features + 2-d Gaze = 514-d State    │
└────────────────────────────────────────┬────────────────────────────────────────┘
                                         │
                                         ▼
┌─────────────────────────────────────────────────────────────────────────────────┐
│ 4. SaliencyFoveatedWrapper                                                      │
│    Gym Environment ObservationWrapper for RL (SAC / PPO)                        │
└────────────────────────────────────────┬────────────────────────────────────────┘
                                         │
                                         ▼
┌─────────────────────────────────────────────────────────────────────────────────┐
│ 5. Test Harness (This File)                                                     │
│    Validation, Telemetry Extraction & Matplotlib Visual Dashboard               │
└─────────────────────────────────────────────────────────────────────────────────┘

```

In [ ]:
import matplotlib.pyplot as plt
from gym_wrapper.saliency_foveation_wrapper import SaliencyFoveatedWrapper
import numpy as np

def test_unbiased_active_vision(env, grabber):
    wrapped_env = SaliencyFoveatedWrapper(env, grabber, crop_size=96)

    # 1. Reset
    state = wrapped_env.reset()
    print(f"Observation space: {wrapped_env.observation_space.shape}")
    print(f"Sampled state shape: {state.shape} (Expected: 514)")

    # 2. Step forward 5 steps
    for step in range(1, 6):
        action = np.array([0.0, 0.5, -1.0])
        state, reward, done, info = wrapped_env.step(action)

        print(f"Step {step} | Peak Fixation: {wrapped_env.latest_gaze}")

        if step == 5:
            fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

            # Full Scene with Fixation Mark
            gx, gy = wrapped_env.latest_gaze
            axes[0].imshow(wrapped_env.latest_frame_rgb)
            axes[0].scatter(
                [gx],
                [gy],
                c="red",
                s=120,
                marker="x",
                linewidths=2.5,
                label=f"Peak Saliency ({gx}, {gy})",
            )
            axes[0].set_title("Input Frame + Saliency Argmax")
            axes[0].legend(loc="lower right")
            axes[0].axis("off")

            # Saliency Heatmap
            axes[1].imshow(wrapped_env.latest_heatmap, cmap="inferno")
            axes[1].set_title("Unbiased Saliency Heatmap")
            axes[1].axis("off")

            # Foveal Crop
            axes[2].imshow(wrapped_env.latest_crop)
            axes[2].set_title("Foveated Crop (Sent to ResNet-18)")
            axes[2].axis("off")

            plt.tight_layout()
            plt.show()

        if done:
            break